#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [1]:
!pip install transformers datasets evaluate accelerate gradio -q
!pip install huggingface_hub -q
import torch
if torch.cuda.is_available():
    print(f"доступенъ GPU: {torch.cuda.get_device_name(0)}")
    import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import load_dataset
import evaluate
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#Загрузите датасет ag_news
dataset = load_dataset('ag_news')
#Выберите модель для дообучения (например, distilbert-base-uncased или bert-base-uncased), num_labels=4
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
).to(device)
#Токенизируйте данные (max_length=128)
def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)
tokenized_datasets = dataset.map(tokenize, batched=True)
train_dataset = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(6000))
#Настройте TrainingArguments...
training_args = TrainingArguments(
    output_dir="./ag_news_model", #Сохраните модель в папку ./ag_news_model
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
доступенъ GPU: Tesla T4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [4]:
#Обучите модель с помощью Trainer. Для метрик используйте evaluate.load("accuracy")
accuracy_metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [6]:
#обученіе прервали на 5-ой минутѣ:
#Выведите accuracy на тестовой выборке
evaccurac = trainer.evaluate()
print(f"\nМетрики по обученію: {evaccurac}")
#Протестируйте модель на трех новых новостях (вписать вручную), используя pipeline. Выведите предсказанный класс и уверенность модели
from transformers import pipeline
classifier = pipeline('text-classification', model=model, tokenizer=tokenizer)
test_texts = [
    'Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Streets dwindling band of ultra-cynics, are seeing green again.',
    'State grant to aid Lynn; Bank gives Salem $10k Central Square in Lynn should be looking a bit brighter. New sidewalks, curbs, fences, lights, landscaping, and road improvements are planned for the Gateway Artisan Block, a key area of the square, with...',
    'Whats in a Name? Well, Matt Is Sexier Than Paul (Reuters) Reuters - As Shakespeare said, a rose by any other name would smell as sweet. Right?'
]
for text in test_texts:
    result = classifier(text)[0]
    print(f"Text: {text}\nClassification: {result['label']}, Score: {result['score']:.4f}\n")

Epoch,Training Loss,Validation Loss
0,0.226644,0.225869



Метрики по обученію: {'eval_loss': 0.22586889564990997}
Text: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Streets dwindling band of ultra-cynics, are seeing green again.
Classification: LABEL_3, Score: 0.7764

Text: State grant to aid Lynn; Bank gives Salem $10k Central Square in Lynn should be looking a bit brighter. New sidewalks, curbs, fences, lights, landscaping, and road improvements are planned for the Gateway Artisan Block, a key area of the square, with...
Classification: LABEL_0, Score: 0.4828

Text: Whats in a Name? Well, Matt Is Sexier Than Paul (Reuters) Reuters - As Shakespeare said, a rose by any other name would smell as sweet. Right?
Classification: LABEL_3, Score: 0.5531

